# LiveOcean Rolodex BestEstimate

Open the public LiveOcean Icechunk store, use Rolodex to extract a
best-estimate time series with a 2-hour forecast offset, and plot
surface temperature.


In [1]:
import numpy as np
import pandas as pd
import icechunk
import xarray as xr
import hvplot.xarray
import holoviews as hv
import rolodex.forecast
from rolodex.forecast import BestEstimate, ForecastIndex

hv.extension("bokeh")


In [2]:
SOURCE_COOP_BUCKET = "us-west-2.opendata.source.coop"
SOURCE_COOP_PREFIX = "rsignell/liveocean/icechunk/liveocean-layers-icechunk-example"
LIVEOCEAN_URL_PREFIX = "s3://liveocean-share/"
LIVEOCEAN_ENDPOINT_URL = "https://s3.kopah.uw.edu"


## Open the public Icechunk store

Both the Source Cooperative Icechunk metadata and the referenced
LiveOcean NetCDF virtual chunks are read anonymously.


In [3]:
storage = icechunk.s3_storage(
    bucket=SOURCE_COOP_BUCKET,
    prefix=SOURCE_COOP_PREFIX,
    region="us-west-2",
    anonymous=True,
)

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=LIVEOCEAN_URL_PREFIX,
        store=icechunk.s3_store(
            region="not-used",
            anonymous=True,
            s3_compatible=True,
            force_path_style=True,
            endpoint_url=LIVEOCEAN_ENDPOINT_URL,
        ),
    )
)

credentials = icechunk.containers_credentials(
    {LIVEOCEAN_URL_PREFIX: icechunk.s3_credentials(anonymous=True)}
)

repo = icechunk.Repository.open(
    storage,
    config,
    authorize_virtual_chunk_access=credentials,
)
session = repo.readonly_session("main")
ds = xr.open_zarr(session.store, consolidated=False, chunks={})
ds


<xarray.Dataset> Size: 73GB
Dimensions:                (time: 7, step: 19, eta_rho: 1302, xi_rho: 663)
Coordinates:
  * time                   (time) datetime64[ns] 56B 2026-05-01 ... 2026-05-07
  * step                   (step) timedelta64[ns] 152B 0 days 00:00:00 ... 3 ...
    lon_rho                (eta_rho, xi_rho) float64 7MB dask.array<chunksize=(1302, 663), meta=np.ndarray>
    lat_rho                (eta_rho, xi_rho) float64 7MB dask.array<chunksize=(1302, 663), meta=np.ndarray>
    ocean_time             datetime64[ns] 8B ...
Dimensions without coordinates: eta_rho, xi_rho
Data variables: (12/86)
    ARAG_10                (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    ARAG_1000              (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    ARAG_100               (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    ARAG_surface           (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    ARAG_1500              (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    ARAG_20                (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    ...                     ...
    temp_30                (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    temp_bottom            (time, step, eta_rho, xi_rho) float32 459MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    temp_50                (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    temp_2500              (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    temp_surface           (time, step, eta_rho, xi_rho) float32 459MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    temp_500               (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
Attributes:
    history:                Fri May  1 05:41:53 2026: ncrcat -p /mmfs1/gscrat...
    NCO:                    netCDF Operators version 5.3.3 (Homepage = http:/...
    nco_input_file_number:  19
    nco_input_file_list:    layers_000000.nc layers_000001.nc layers_000002.n...

## Extract a Rolodex BestEstimate dataset

Rolodex indexes the forecast reference time, forecast step, and
valid time so a single monotonic best-estimate series can be
selected. Here the selected offset is 2 hours.


In [4]:
ds.coords["valid_time"] = rolodex.forecast.create_lazy_valid_time_variable(
    reference_time=ds.time,
    period=ds.step,
)

fmrc = ds.drop_indexes(["time", "step"]).set_xindex(
    ["time", "step", "valid_time"],
    ForecastIndex,
)

ds_best = fmrc.sel(valid_time=BestEstimate(offset=2))
ds_best


<xarray.Dataset> Size: 30GB
Dimensions:                (valid_time: 53, eta_rho: 1302, xi_rho: 663)
Coordinates:
  * valid_time             (valid_time) datetime64[ns] 424B 2026-05-01T08:00:...
    time                   (valid_time) datetime64[ns] 424B 2026-05-01 ... 20...
    step                   (valid_time) timedelta64[ns] 424B 0 days 08:00:00 ...
    lon_rho                (eta_rho, xi_rho) float64 7MB dask.array<chunksize=(1302, 663), meta=np.ndarray>
    lat_rho                (eta_rho, xi_rho) float64 7MB dask.array<chunksize=(1302, 663), meta=np.ndarray>
    ocean_time             datetime64[ns] 8B ...
Dimensions without coordinates: eta_rho, xi_rho
Data variables: (12/86)
    ARAG_10                (valid_time, eta_rho, xi_rho) float64 366MB dask.array<chunksize=(1, 1302, 663), meta=np.ndarray>
    ARAG_1000              (valid_time, eta_rho, xi_rho) float64 366MB dask.array<chunksize=(1, 1302, 663), meta=np.ndarray>
    ARAG_100               (valid_time, eta_rho, xi_rho) float64 366MB dask.array<chunksize=(1, 1302, 663), meta=np.ndarray>
    ARAG_surface           (valid_time, eta_rho, xi_rho) float64 366MB dask.array<chunksize=(1, 1302, 663), meta=np.ndarray>
    ARAG_1500              (valid_time, eta_rho, xi_rho) float64 366MB dask.array<chunksize=(1, 1302, 663), meta=np.ndarray>
    ARAG_20                (valid_time, eta_rho, xi_rho) float64 366MB dask.array<chunksize=(1, 1302, 663), meta=np.ndarray>
    ...                     ...
    temp_30                (valid_time, eta_rho, xi_rho) float64 366MB dask.array<chunksize=(1, 1302, 663), meta=np.ndarray>
    temp_bottom            (valid_time, eta_rho, xi_rho) float32 183MB dask.array<chunksize=(1, 1302, 663), meta=np.ndarray>
    temp_50                (valid_time, eta_rho, xi_rho) float64 366MB dask.array<chunksize=(1, 1302, 663), meta=np.ndarray>
    temp_2500              (valid_time, eta_rho, xi_rho) float64 366MB dask.array<chunksize=(1, 1302, 663), meta=np.ndarray>
    temp_surface           (valid_time, eta_rho, xi_rho) float32 183MB dask.array<chunksize=(1, 1302, 663), meta=np.ndarray>
    temp_500               (valid_time, eta_rho, xi_rho) float64 366MB dask.array<chunksize=(1, 1302, 663), meta=np.ndarray>
Attributes:
    history:                Fri May  1 05:41:53 2026: ncrcat -p /mmfs1/gscrat...
    NCO:                    netCDF Operators version 5.3.3 (Homepage = http:/...
    nco_input_file_number:  19
    nco_input_file_list:    layers_000000.nc layers_000001.nc layers_000002.n...

## Surface temperature at the last valid time


In [8]:
def format_time(value):
    return pd.Timestamp(value).strftime("%Y-%m-%d %H:%M:%S UTC")


last_temp = ds_best["temp_surface"].isel(valid_time=-1)
last_valid_time = ds_best.valid_time.values[-1]

surface_map = last_temp.hvplot.quadmesh(
    x="lon_rho",
    y="lat_rho",
    geo=True,
    tiles="OSM",
    rasterize=True,
    cmap="turbo",
    width=900,
    height=650,
    title=f"Surface temperature | valid: {format_time(last_valid_time)}",
)
surface_map


:DynamicMap   []
   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [lon_rho,lat_rho]   (potential temperature at surface)

## Time series in Puget Sound


In [6]:
def nearest_lon_lat_indices(lon, lat, target_lon, target_lat):
    distance2 = (lon - target_lon) ** 2 + (lat - target_lat) ** 2
    flat_index = int(np.nanargmin(distance2.values))
    return tuple(
        int(index) for index in np.unravel_index(flat_index, lon.shape)
    )


target_lon = -122.45
target_lat = 47.65

eta_index, xi_index = nearest_lon_lat_indices(
    ds_best.lon_rho,
    ds_best.lat_rho,
    target_lon=target_lon,
    target_lat=target_lat,
)

selected_lon = float(ds_best.lon_rho.isel(eta_rho=eta_index, xi_rho=xi_index))
selected_lat = float(ds_best.lat_rho.isel(eta_rho=eta_index, xi_rho=xi_index))

eta_index, xi_index, selected_lon, selected_lat


(754, 595, -122.45293354776767, 47.65111173195382)

In [9]:
temp_series = ds_best["temp_surface"].isel(
    eta_rho=eta_index,
    xi_rho=xi_index,
)

temp_series.hvplot(
    x="valid_time",
    grid=True,
    width=900,
    height=350,
    title=(
        "Surface temperature near Puget Sound "
        f"({selected_lon:.3f}, {selected_lat:.3f})"
    ),
)


:Curve   [valid_time]   (potential temperature at surface)